# 23a — Fisher uncertainties for the exp6 `trial_07` model on the 14 real experiments

**Series 23 opener.** The model under study is the **best KDE-based model on the real data**:
the AG-HYPOPT **experiment_6 `trial_07`** score design (the exp5 winning schedule + the exp6 winner
`sigma_weight 0.8649`, `h_f_scale 4.0`, `h_s_scale 3.6745`, `gamma_rel_cap 0.10736`), i.e. the exact
series-21 μ mechanism (Anuar's `loglik_mean` reward, `sigma_prop` score, **no clip, no clamps**,
init `0.5 x truth`, `n_runs=100`, `n_iter=30`).

**Why KDE.** Fisher/Cramér-Rao needs a likelihood. The 2-D `(FWHM, sigma_fit)` **KDE** is our likelihood;
the exp8 alternative losses (Wasserstein, CvM, energy, MMD) are discrepancies and have no CRB. So the
KDE-only model is the right base for an uncertainty analysis.

**What this notebook does.** For **all 14 real experiments**:
1. runs the frozen optimisation (deterministic), records the full path + per-step simulated cloud;
2. at the optimum computes the **Fisher information of the KDE likelihood** and the CRB uncertainties
   `sigma_mu`, `sigma_gamma` (and their correlation) under the **existing convention** (the per-scan
   normalisation, `J = sum_i s_i s_i^T / N`) **and** under the two corrections discussed:
   - `sigma_data = sigma_single / sqrt(n_target)` — the dataset CRB (undo the `/N`);
   - the **Scott-baseline** bandwidth (`h_f_scale = h_s_scale = 1.0`), to separate the UQ from the
     bandwidth that exp6/7 inflated to fix the mu floor.

**Not touched.** No change to `src/` or to any previous notebook; this notebook copies the exp6 helpers
it needs. Figures render **inline only** (no savefig); executed **in place**.

## Panel (series-22 visual conventions)
- **FIG 1 (interactive, Plotly)** — per experiment: the **(mu, gamma) parameter path** (left) and the
  **generated (FWHM, sigma_fit) cloud at that step** (right) against the real data. Experiment dropdown +
  play button / slider over the iteration.
- **FIG 2 — the uncertainty report, split by power: left = 1nW, right = 3nW.** Top row: `mu_hat/mu_true`
  with the CRB `sigma_mu` whiskers (three variants: per-scan, dataset-corrected, Scott-baseline);
  bottom row: same for gamma. Shows both the point estimate and the uncertainty, which is the point.
- **FIG 3** normalised trajectories `mu/mu_true`, `gamma/gamma_true` vs iteration (per power).
- **FIG 4** accuracy vs transmission (mu and gamma ratios, both powers, goal bands).

In [1]:
# ============================================================
# 23a — imports (repo bootstrap like the 22-series)
# ============================================================
import math, os, sys, time, json
from contextlib import nullcontext
import numpy as np
import torch
import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor as _PPE

torch.set_default_dtype(torch.float32)
for _p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    if os.path.isdir(os.path.join(_p, 'src')):
        sys.path.insert(0, _p); REPO_ROOT = _p; break
os.chdir(REPO_ROOT)

from src.fitting import fit_profile, fwhm_from_theta, nll
from src.samplers import draw_fixed_noise
from src.implicit import compute_fwhm_and_dgamma

import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.templates.default = 'plotly_white'
pio.renderers.default = os.environ.get('PLOTLY_RENDERER', 'vscode')
print('Imports OK | plotly', plotly.__version__, '| repo', REPO_ROOT)


Imports OK | plotly 7.1.0 | repo /home/pukky/.openclaw/workspace/qm-ml


In [2]:
# ============================================================
# 23a — EXPERIMENTS (14 real; true values from Gregor's fits)
# ============================================================
EXPERIMENTS = [
    dict(name='1nW Trans05',  power='1nW', mu_true=9.393,   sigma_prop=2.576,  lam=2.232, gamma_true=8.5,  n_target=61),
    dict(name='1nW Trans10',  power='1nW', mu_true=12.372,  sigma_prop=3.445,  lam=2.122, gamma_true=8.5,  n_target=358),
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316,  sigma_prop=4.141,  lam=2.286, gamma_true=8.5,  n_target=1138),
    dict(name='1nW Trans40',  power='1nW', mu_true=38.405,  sigma_prop=7.198,  lam=2.351, gamma_true=8.5,  n_target=2428),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374,  sigma_prop=9.851,  lam=2.593, gamma_true=8.5,  n_target=2424),
    dict(name='1nW Trans80',  power='1nW', mu_true=79.365,  sigma_prop=12.627, lam=2.758, gamma_true=8.5,  n_target=2487),
    dict(name='1nW Trans100', power='1nW', mu_true=70.817,  sigma_prop=17.221, lam=2.636, gamma_true=8.5,  n_target=2455),
    dict(name='3nW Trans05',  power='3nW', mu_true=13.204,  sigma_prop=3.724,  lam=2.186, gamma_true=14.1, n_target=252),
    dict(name='3nW Trans10',  power='3nW', mu_true=24.476,  sigma_prop=5.639,  lam=2.158, gamma_true=14.1, n_target=1572),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279,  sigma_prop=8.319,  lam=2.264, gamma_true=14.1, n_target=2171),
    dict(name='3nW Trans40',  power='3nW', mu_true=84.892,  sigma_prop=24.013, lam=2.475, gamma_true=14.1, n_target=3742),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95,  lam=2.741, gamma_true=14.1, n_target=2541),
    dict(name='3nW Trans80',  power='3nW', mu_true=137.537, sigma_prop=32.107, lam=2.911, gamma_true=14.1, n_target=2508),
    dict(name='3nW Trans100', power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516),
]
POWERS = ['1nW', '3nW']
TRANS  = ['05', '10', '20', '40', '60', '80', '100']
print(len(EXPERIMENTS), 'experiments')


14 experiments


In [3]:
# ============================================================
# 23a CONFIG — FROZEN experiment_6 trial_07 model + the Fisher knobs
# ============================================================
CFG = dict(
    n_runs=100, n_iter=30,                       # series-21/exp5/6 protocol
    # FROZEN schedule = experiment_5 winning trial (trial_21)
    lr_mu=0.1669, mu_anneal=0.3455, lr_gamma=0.4716, gamma_anneal=0.4723,
    sigma_ref=10.0, clip=float('inf'),           # no gradient clipping (21i)
    # FROZEN experiment_6 winner (trial_07)
    sigma_weight=0.8648862719598797,             # sigma-channel damping
    gamma_rel_cap=0.10735617789825944,           # relative gamma trust region
    h_f_scale=4.0, h_s_scale=3.674548492934102,  # inflated Scott bandwidths
    h_s_min=0.05,
)
H_REF = 1.0            # z-form gamma-score reference (optimizer only)
SEED  = 42             # per-step noise seed base (deterministic)
REAL_CSV = os.path.join(REPO_ROOT, 'data', 'processed', 'fwhm_linewidths.csv')

# ---- Fisher knobs ----
M_FINAL          = 500     # simulated scans for the Fisher estimate (per seed)
FISHER_SEEDS     = 5       # independent seeds averaged (as 12d)
FISHER_BASE_SEED = 7000
SCOTT_SCALES     = (1.0, 1.0)   # the un-inflated (Scott) bandwidth baseline

OUT_JSON = os.path.join(REPO_ROOT, 'data', 'processed', '23a_fisher_exp6_trial07.json')

SMOKE = os.environ.get('NB_SMOKE') == '1'
if SMOKE:
    CFG['n_runs'], CFG['n_iter'] = 20, 4
    EXPERIMENTS = EXPERIMENTS[4:5]
    M_FINAL, FISHER_SEEDS = 40, 1
print('config: %s | M_FINAL=%d, seeds=%d | SMOKE=%s' % (CFG, M_FINAL, FISHER_SEEDS, SMOKE))


config: {'n_runs': 100, 'n_iter': 30, 'lr_mu': 0.1669, 'mu_anneal': 0.3455, 'lr_gamma': 0.4716, 'gamma_anneal': 0.4723, 'sigma_ref': 10.0, 'clip': inf, 'sigma_weight': 0.8648862719598797, 'gamma_rel_cap': 0.10735617789825944, 'h_f_scale': 4.0, 'h_s_scale': 3.674548492934102, 'h_s_min': 0.05} | M_FINAL=500, seeds=5 | SMOKE=False


In [4]:
# ============================================================
# 23a — helpers (verbatim from experiment_6/ag_hypopt.py, plus a gamma_scale switch)
# ============================================================
GAMMA_SCALE = True     # optimizer uses the z-form gamma-score; the Fisher sets this False

def _kde_scores(sim_f, sim_s, sim_n, sim_df, sim_ds, data_f, data_s, h_f, h_s, mu, sigma_prop, cfg):
    '''2-D KDE log-likelihood pieces + per-DATA-POINT scores (as exp5/6).'''
    sw = float(cfg.get('sigma_weight', 1.0))
    d_f = data_f[:, None] - sim_f[None, :]
    d_s = data_s[:, None] - sim_s[None, :]
    W = torch.exp(-0.5 * (d_f / h_f) ** 2 - 0.5 * sw * (d_s / h_s) ** 2)
    w = W / W.sum(dim=1, keepdim=True).clamp_min(1e-12)
    score = (sim_n[None, :] - mu) / sigma_prop ** 2
    s_mu = (w * score).sum(dim=1)
    if GAMMA_SCALE:
        dlogG = ((d_f * sim_df[None, :]) / h_f + sw * (d_s * sim_ds[None, :]) / h_s) / H_REF
    else:
        dlogG = (d_f * sim_df[None, :]) / h_f ** 2 + sw * (d_s * sim_ds[None, :]) / h_s ** 2
    s_gamma = (w * dlogG).sum(dim=1)
    logp = torch.log((W.sum(dim=1) / len(sim_f)).clamp_min(1e-30))
    return s_mu, s_gamma, -logp.mean(), w, W

def _fit_fn(ph):  return fit_profile(ph, n_iters=80, model='lorentzian', uniform_bg=False)
def _fwhm_fn(th): return fwhm_from_theta(th, model='lorentzian')
def _nll_fn(th, ph): return nll(th, ph, model='lorentzian', uniform_bg=False)

def _run_one(args):
    gamma_val, u, b = args
    return compute_fwhm_and_dgamma(gamma_val, u, b, _fit_fn, _fwhm_fn, _nll_fn, n_params=2)

def _init_worker(): torch.set_num_threads(1)
def _parallel_map(pool, tasks): return list(pool.map(_run_one, tasks, chunksize=8))

def _load_real_target(exp):
    '''Real measured target for `exp` (15-series convention): raw*1000 -> MHz, keep err/fwhm < 10.'''
    import pandas as pd
    power_nW = int(str(exp['power']).replace('nW', ''))
    trans = int(str(exp['name']).split('Trans')[-1])
    df = pd.read_csv(REAL_CSV)
    sub = df[(df['power_nW'] == power_nW) & (df['transmission'] == trans)]
    f = sub['fwhm'].to_numpy(dtype=float) * 1000.0
    e = sub['fit_error'].to_numpy(dtype=float) * 1000.0
    with np.errstate(invalid='ignore', divide='ignore'):
        ok = np.isfinite(f) & np.isfinite(e) & (f > 0)
        filt = ok & ((e / f) < 10.0)
    return (torch.tensor(f[filt], dtype=torch.float32), torch.tensor(e[filt], dtype=torch.float32))

def _bandwidths(target_f, target_s, h_f_scale, h_s_scale, h_s_min):
    '''Scott-from-target, scaled (FIXED w.r.t. mu,gamma -> a clean likelihood).'''
    scott = len(target_f) ** (-1.0 / 6.0)
    H_F = h_f_scale * float(target_f.std()) * scott
    H_S = max(h_s_scale * float(target_s.std()) * scott, h_s_min)
    return H_F, H_S

def _sims(pool, mu, sigma_prop, lam, gamma, n_runs, seed):
    rng = np.random.default_rng(seed)
    tasks, ns = [], []
    for _ in range(n_runs):
        u, b, n = draw_fixed_noise(mu, sigma_prop, lam, rng)
        tasks.append((gamma, u.numpy(), b.numpy())); ns.append(n)
    res = _parallel_map(pool, tasks)
    return (torch.tensor([r[0] for r in res], dtype=torch.float32),
            torch.tensor([r[1] for r in res], dtype=torch.float32),
            torch.tensor(ns, dtype=torch.float32),
            torch.tensor([r[2] for r in res], dtype=torch.float32),
            torch.tensor([r[3] for r in res], dtype=torch.float32))

def _fisher_at(pool, mu, gamma, sigma_prop, lam, target_f, target_s, H_F, H_S, cfg):
    '''Fisher of the KDE likelihood at (mu,gamma): per-scan J and the data-scaled J.
       Returns dict with sigma_mu/sigma_gamma under the per-scan normalisation and the
       dataset correction sigma/sqrt(N).'''
    global GAMMA_SCALE
    gs = GAMMA_SCALE; GAMMA_SCALE = False          # Fisher uses the RAW likelihood chain
    N = int(len(target_f))
    Js = []
    try:
        for s_i in range(FISHER_SEEDS):
            ft, si_t, nt, dg_t, ds_t = _sims(pool, mu, sigma_prop, lam, gamma, M_FINAL, FISHER_BASE_SEED + s_i)
            s_mu, s_gamma, _, _, _ = _kde_scores(ft, si_t, nt, dg_t, ds_t,
                                                 target_f, target_s, H_F, H_S, mu, sigma_prop, cfg)
            s = torch.stack([s_mu, s_gamma], dim=1)
            Js.append(s.T @ s / N)                 # the EXISTING convention (per-scan)
    finally:
        GAMMA_SCALE = gs
    J_single = torch.stack(Js).mean(dim=0)
    inv_single = torch.linalg.inv(J_single + 1e-12 * torch.eye(2))
    inv_data = inv_single / N                      # dataset CRB = (N*J)^-1
    out = dict(N=N, J=J_single.tolist(),
               sig_mu_single=float(math.sqrt(inv_single[0, 0])),
               sig_g_single=float(math.sqrt(inv_single[1, 1])),
               corr_single=float(inv_single[0, 1] / math.sqrt(inv_single[0, 0] * inv_single[1, 1])),
               sig_mu_data=float(math.sqrt(inv_data[0, 0])),
               sig_g_data=float(math.sqrt(inv_data[1, 1])),
               corr_data=float(inv_data[0, 1] / math.sqrt(inv_data[0, 0] * inv_data[1, 1])))
    return out

print('helpers ready')


helpers ready


In [5]:
# ============================================================
# 23a — one experiment: the FROZEN optimisation (exp6 mechanism), recording the path
# ============================================================
def run_experiment(exp, cfg, pool):
    mu_true, sigma_prop = exp['mu_true'], exp['sigma_prop']
    lam, gamma_true = exp['lam'], exp['gamma_true']
    mu_init, gamma_init = 0.5 * mu_true, 0.5 * gamma_true
    n_runs, n_iter = cfg['n_runs'], cfg['n_iter']
    lr_mu, lr_gamma = cfg['lr_mu'], cfg['lr_gamma']
    mu_anneal, gamma_anneal = cfg['mu_anneal'], cfg['gamma_anneal']
    gamma_rel_cap = cfg['gamma_rel_cap']; clip = cfg['clip']

    target_f, target_s = _load_real_target(exp)
    H_F, H_S = _bandwidths(target_f, target_s, cfg['h_f_scale'], cfg['h_s_scale'], cfg['h_s_min'])

    mu_val, gamma_val = float(mu_init), float(gamma_init)
    history, diverged, diverged_at = [], False, None
    t0 = time.time()
    for step in range(n_iter):
        ft, si_t, nt, dg_t, ds_t = _sims(pool, mu_val, sigma_prop, lam, gamma_val, n_runs, SEED + step)
        _, s_gamma, nll_val, w, W = _kde_scores(ft, si_t, nt, dg_t, ds_t,
                                                target_f, target_s, H_F, H_S, mu_val, sigma_prop, cfg)
        r = torch.log(W.clamp_min(1e-30)).mean(dim=0)                 # Anuar's reward
        score = (nt - mu_val) / sigma_prop ** 2                       # sigma_prop score
        grad_mu = float(-(r - r.mean()) @ score)
        grad_gamma = float(-s_gamma.mean())
        mu_val -= lr_mu * (1.0 - mu_anneal * step / n_iter) * grad_mu
        dgamma = lr_gamma * (1.0 - gamma_anneal * step / n_iter) * grad_gamma
        lim = gamma_rel_cap * abs(gamma_val)
        dgamma = max(-lim, min(lim, dgamma))
        gamma_val -= dgamma
        history.append(dict(step=step, mu=float(mu_val), gamma=float(gamma_val), nll=float(nll_val),
                            grad_mu=grad_mu, grad_gamma=grad_gamma,
                            f=[round(float(x), 3) for x in ft], s=[round(float(x), 3) for x in si_t]))
        if not (0.2 <= mu_val <= 1500.0) or not (0.02 <= gamma_val <= 500.0):
            diverged = True; diverged_at = step; break
    return dict(exp=exp['name'], power=exp['power'], mu_true=mu_true, gamma_true=gamma_true,
                sigma_prop=sigma_prop, lam=lam, n_target=int(len(target_f)),
                mu_init=mu_init, gamma_init=gamma_init,
                mu_final=history[-1]['mu'], gamma_final=history[-1]['gamma'],
                nll_final=history[-1]['nll'], history=history,
                target_f=[round(float(x), 3) for x in target_f], target_s=[round(float(x), 3) for x in target_s],
                H_F=H_F, H_S=H_S, diverged=diverged, diverged_at=diverged_at,
                t_elapsed=time.time() - t0)

print('run_experiment ready')


run_experiment ready


In [6]:
# ============================================================
# 23a — RUN: all 14 real experiments (the long cell)
# ============================================================
N_WORKERS = 4
CACHED = (not SMOKE) and os.path.exists(OUT_JSON) and os.environ.get('NB_RECOMPUTE') != '1'
RESULTS = []
if CACHED:
    RESULTS = json.load(open(OUT_JSON))['results']
    print(f'loaded {len(RESULTS)} experiments from {OUT_JSON} (set NB_RECOMPUTE=1 to recompute)')
t0 = time.time()
with (nullcontext() if CACHED else _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'), initializer=_init_worker)) as pool:
    for exp in EXPERIMENTS:
        if CACHED:
            continue
        r = run_experiment(exp, CFG, pool)
        RESULTS.append(r)
        print(f"  {r['exp']:13s} mu {r['mu_true']:7.2f} -> {r['mu_final']:7.2f} | "
              f"gamma {r['gamma_true']:5.2f} -> {r['gamma_final']:6.2f} | NLL {r['nll_final']:.3f} | "
              f"{r['t_elapsed']/60:.1f} min{'  DIVERGED' if r['diverged'] else ''}", flush=True)
print(f'\ntotal {len(RESULTS)} experiments in {(time.time()-t0)/60:.1f} min')


loaded 14 experiments from /home/pukky/.openclaw/workspace/qm-ml/data/processed/23a_fisher_exp6_trial07.json (set NB_RECOMPUTE=1 to recompute)

total 14 experiments in 0.0 min


In [7]:
# ============================================================
# 23a — FISHER at each optimum, three variants:
#   (A) frozen bandwidths, EXISTING per-scan normalisation  -> sigma_*_single
#   (B) frozen bandwidths, dataset correction  sig/sqrt(N)  -> sigma_*_data
#   (C) Scott baseline (h=1),     dataset correction        -> sigma_*_data_scott
# ============================================================
t0 = time.time()
if CACHED:
    # Fisher numbers are already aggregated in the cached table; skip the heavy block.
    for r in RESULTS:
        r['fisher_frozen'] = {'sig_mu_single': None}
    print('fisher block: cached (numbers below come from the saved table)')
with (nullcontext() if CACHED else _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'), initializer=_init_worker)) as pool:
    for r in ([] if CACHED else RESULTS):
        exp = next(e for e in EXPERIMENTS if e['name'] == r['exp'])
        tf = torch.tensor(r['target_f'], dtype=torch.float32)
        ts = torch.tensor(r['target_s'], dtype=torch.float32)
        # (A)/(B): the frozen (inflated) bandwidths
        r['fisher_frozen'] = _fisher_at(pool, r['mu_final'], r['gamma_final'], r['sigma_prop'],
                                        r['lam'], tf, ts, r['H_F'], r['H_S'], CFG)
        # (C): the Scott baseline bandwidths
        H_F0, H_S0 = _bandwidths(tf, ts, *SCOTT_SCALES, CFG['h_s_min'])
        r['fisher_scott'] = _fisher_at(pool, r['mu_final'], r['gamma_final'], r['sigma_prop'],
                                       r['lam'], tf, ts, H_F0, H_S0, CFG)
        print(f"  {r['exp']:13s} N={r['n_target']:5d} | frozen: sig_mu {r['fisher_frozen']['sig_mu_single']:8.1f}"
              f" -> {r['fisher_frozen']['sig_mu_data']:7.2f} | scott: sig_mu {r['fisher_scott']['sig_mu_data']:7.2f}", flush=True)
print(f'fisher block: {(time.time()-t0)/60:.1f} min')


fisher block: cached (numbers below come from the saved table)
fisher block: 0.0 min


In [8]:
# ============================================================
# 23a — the uncertainty table + coverage (|Delta|/sigma) under each variant
# ============================================================
def _row(r):
    fr, sc = r['fisher_frozen'], r['fisher_scott']
    dmu = abs(r['mu_final'] - r['mu_true']); dg = abs(r['gamma_final'] - r['gamma_true'])
    return dict(exp=r['exp'], n=r['n_target'], mu_true=r['mu_true'], mu=r['mu_final'],
                gamma_true=r['gamma_true'], gamma=r['gamma_final'],
                sig_mu_single=fr['sig_mu_single'], sig_mu_data=fr['sig_mu_data'],
                sig_mu_data_scott=sc['sig_mu_data'],
                sig_g_single=fr['sig_g_single'], sig_g_data=fr['sig_g_data'],
                sig_g_data_scott=sc['sig_g_data'],
                corr_frozen=fr['corr_data'], corr_scott=sc['corr_data'],
                dmu_sigma_single=dmu / fr['sig_mu_single'], dmu_sigma_data=dmu / fr['sig_mu_data'],
                dmu_sigma_data_scott=dmu / sc['sig_mu_data'],
                dg_sigma_single=dg / fr['sig_g_single'], dg_sigma_data=dg / fr['sig_g_data'],
                dg_sigma_data_scott=dg / sc['sig_g_data'])

TABLE = json.load(open(OUT_JSON))['table'] if CACHED else [_row(r) for r in RESULTS]
print(f'{"exp":13s} {"N":>5s} | {"sig_mu single":>13s} {"data":>8s} {"data+scott":>10s} | '
      f'{"|dmu|/sig single":>15s} {"data":>7s} {"scott":>7s} | {"2sig(single)":>12s}')
for t in TABLE:
    ok = 'YES' if t['dmu_sigma_single'] <= 2 else 'no'
    print(f'{t["exp"]:13s} {t["n"]:5d} | {t["sig_mu_single"]:13.2f} {t["sig_mu_data"]:8.2f} '
          f'{t["sig_mu_data_scott"]:10.2f} | {t["dmu_sigma_single"]:15.2f} {t["dmu_sigma_data"]:7.2f} '
          f'{t["dmu_sigma_data_scott"]:7.2f} | {ok:>12s}')
cov = lambda k, thr=2.0: sum(1 for t in TABLE if t[k] <= thr)
print()
print('coverage within 2 sigma:  per-scan %d/14 | dataset %d/14 | dataset+scott %d/14'
      % (cov('dmu_sigma_single'), cov('dmu_sigma_data'), cov('dmu_sigma_data_scott')))
print('coverage within 1 sigma (mu):  per-scan %d/14 | dataset %d/14 | dataset+scott %d/14'
      % (cov('dmu_sigma_single', 1), cov('dmu_sigma_data', 1), cov('dmu_sigma_data_scott', 1)))
print()
print('gamma: median |dgamma|/sigma  per-scan %.2f | dataset %.2f | dataset+scott %.2f'
      % (np.median([t['dg_sigma_single'] for t in TABLE]),
         np.median([t['dg_sigma_data'] for t in TABLE]),
         np.median([t['dg_sigma_data_scott'] for t in TABLE])))


exp               N | sig_mu single     data data+scott | |dmu|/sig single    data   scott | 2sig(single)


1nW Trans05      61 |         38.92     4.98       3.35 |            0.09    0.67    0.99 |          YES
1nW Trans10     358 |         51.26     2.71       1.03 |            0.05    0.97    2.55 |          YES
1nW Trans20    1138 |         90.89     2.69       0.38 |            0.11    3.86   27.05 |          YES
1nW Trans40    2428 |        109.29     2.22       0.40 |            0.06    3.17   17.54 |          YES
1nW Trans60    2424 |        100.97     2.05       0.40 |            0.17    8.54   43.71 |          YES
1nW Trans80    2487 |        142.77     2.86       0.65 |            0.22   11.20   49.04 |          YES
1nW Trans100   2455 |         81.37     1.64       0.66 |            0.24   11.65   28.89 |          YES
3nW Trans05     252 |         35.69     2.25       1.11 |            0.08    1.25    2.55 |          YES
3nW Trans10    1572 |         67.92     1.71       0.38 |            0.03    1.29    5.76 |          YES
3nW Trans20    2171 |         81.14     1.74       0.4

In [9]:
# ============================================================
# 23a — save companion run data (paths + per-step clouds + Fisher)
# ============================================================
if SMOKE or CACHED:
    print('smoke/cached: not re-saving')
if not SMOKE and not CACHED:
    payload = dict(notebook='23a-fisher-uncertainties-exp6-trial07',
                   model='AG-HYPOPT experiment_6 trial_07 (KDE, frozen exp5 schedule)',
                   config=CFG, M_FINAL=M_FINAL, FISHER_SEEDS=FISHER_SEEDS,
                   variants=dict(A='frozen bandwidths, per-scan J=sum/N (existing convention)',
                                 B='frozen bandwidths, dataset correction sig/sqrt(N)',
                                 C='Scott baseline h=1, dataset correction'),
                   table=TABLE,
                   results=[{k: v for k, v in r.items() if k not in ('fisher_frozen', 'fisher_scott')} | 
                            dict(fisher_frozen=r['fisher_frozen'], fisher_scott=r['fisher_scott']) for r in RESULTS])
    json.dump(payload, open(OUT_JSON, 'w'))
    print('saved', OUT_JSON, f'({os.path.getsize(OUT_JSON)/1e6:.2f} MB)')
else:
    print('smoke: not saving')


smoke/cached: not re-saving
smoke: not saving


In [10]:
# ============================================================
# 23a — FIG 1 (interactive): (mu,gamma) path + the simulated cloud at each step vs real data
#   experiment dropdown + play/pause + iteration slider
# ============================================================
def _cloud(f, s, xr=(0.0, 70.0), yr=(0.0, 40.0), nx=40, ny=40):
    f = np.asarray(f, float); s = np.asarray(s, float)
    ok = np.isfinite(f) & np.isfinite(s) & (f >= xr[0]) & (f <= xr[1]) & (s >= yr[0]) & (s <= yr[1])
    from scipy.ndimage import gaussian_filter
    H, xe, ye = np.histogram2d(f[ok], s[ok], bins=[nx, ny], range=[xr, yr])
    Z = gaussian_filter(H.T, 1.1)
    Z = Z / max(Z.max(), 1e-12)
    return 0.5 * (xe[:-1] + xe[1:]), 0.5 * (ye[:-1] + ye[1:]), Z

X_RANGE, Y_RANGE = (0.0, 70.0), (0.0, 40.0)
REAL_CAP = 200          # real points per frame (size guard; the JSON keeps the full sets)

def _dec(f, s, cap=REAL_CAP, seed=SEED):
    f = np.asarray(f, float); s = np.asarray(s, float)
    ok = np.isfinite(f) & np.isfinite(s)
    f, s = f[ok], s[ok]
    if f.size > cap:
        rng = np.random.default_rng(seed)
        idx = np.sort(rng.choice(f.size, size=cap, replace=False))
        f, s = f[idx], s[idx]
    return f, s

def _cloud_trace(xc, yc, Z):
    return go.Heatmap(x=xc, y=yc, z=np.round(Z * 255).astype(int), colorscale='Blues',
                      showscale=False, xaxis='x2', yaxis='y2', showlegend=False)

frames, slider_steps = [], []
for ei, r in enumerate(RESULTS):
    rf, rs = _dec(r['target_f'], r['target_s'])
    for k, h in enumerate(r['history']):
        nm = f'{ei}-{k}'
        mus = [hh['mu'] for hh in r['history'][:k + 1]]
        gams = [hh['gamma'] for hh in r['history'][:k + 1]]
        xc, yc, Z = _cloud(h['f'], h['s'], X_RANGE, Y_RANGE)
        # traces [0,1,2,3] = path(x,y) | truth(x,y) | cloud(x2,y2) | real data(x2,y2)
        frames.append(dict(name=nm, traces=[0, 1, 2, 3], data=[
            go.Scatter(x=mus, y=gams, mode='lines+markers', line=dict(color='#1d3557', width=2),
                       marker=dict(size=4), showlegend=False),
            go.Scatter(x=[r['mu_true']], y=[r['gamma_true']], mode='markers',
                       marker=dict(symbol='x', size=12, color='#2a9d8f'), showlegend=False),
            _cloud_trace(xc, yc, Z),
            go.Scatter(x=rf, y=rs, mode='markers', marker=dict(size=2.5, color='#e63946', opacity=0.6),
                       xaxis='x2', yaxis='y2', showlegend=False),
        ]))
        slider_steps.append(dict(method='animate', label=f"{r['exp']} step{k}",
                                 args=[[nm], dict(mode='immediate', frame=dict(duration=0, redraw=True),
                                                  transition=dict(duration=0))]))
r0 = RESULTS[0]; h0 = r0['history'][0]
rf0, rs0 = _dec(r0['target_f'], r0['target_s'])
xc0, yc0, Z0 = _cloud(h0['f'], h0['s'], X_RANGE, Y_RANGE)
fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58], horizontal_spacing=0.09,
                    subplot_titles=['(mu, gamma) path', '(FWHM, sigma_fit) simulated cloud vs real data'])
fig.add_trace(go.Scatter(x=[h0['mu']], y=[h0['gamma']], mode='lines+markers',
                         line=dict(color='#1d3557', width=2), marker=dict(size=4), showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=[r0['mu_true']], y=[r0['gamma_true']], mode='markers',
                         marker=dict(symbol='x', size=12, color='#2a9d8f'), name='truth'), 1, 1)
fig.add_trace(_cloud_trace(xc0, yc0, Z0), 1, 2)
fig.add_trace(go.Scatter(x=rf0, y=rs0, mode='markers', marker=dict(size=2.5, color='#e63946', opacity=0.6),
                         name='real data'), 1, 2)
fig.frames = frames
fig.update_xaxes(range=X_RANGE, title_text='FWHM (MHz)', row=1, col=2)
fig.update_yaxes(range=Y_RANGE, title_text='sigma_fit (MHz)', row=1, col=2)
fig.update_xaxes(title_text='mu (photons)', row=1, col=1); fig.update_yaxes(title_text='gamma (MHz)', row=1, col=1)
fig.update_layout(height=470, margin=dict(t=70, b=40),
                  updatemenus=[dict(type='buttons', showactive=False, x=1.0, y=1.15,
                                    buttons=[dict(label='play', method='animate',
                                                  args=[None, dict(frame=dict(duration=120, redraw=True),
                                                                   fromcurrent=True, transition=dict(duration=0))]),
                                             dict(label='pause', method='animate',
                                                  args=[[None], dict(mode='immediate', frame=dict(duration=0, redraw=False))])])],
                  sliders=[dict(active=0, x=0.0, y=-0.18, len=1.0, currentvalue=dict(prefix=''), steps=slider_steps)])
CONFIG_PLOT = {'scrollZoom': True, 'displaylogo': False, 'responsive': True}
print('FIG 1 frames:', len(frames))
HTML1 = os.path.join(REPO_ROOT, 'notebooks', '23_fisher_uncertainties', '23a-fig1-paths-clouds.html')
fig.write_html(HTML1, include_plotlyjs='cdn', config=CONFIG_PLOT)
print('wrote', HTML1, f'({os.path.getsize(HTML1)/1e6:.1f} MB)')
fig.show(config=CONFIG_PLOT)


FIG 1 frames: 420


wrote /home/pukky/.openclaw/workspace/qm-ml/notebooks/23_fisher_uncertainties/23a-fig1-paths-clouds.html (4.8 MB)


In [11]:
# ============================================================
# 23a — FIG 2a: point estimate +- the CRB whisker (dataset CRB: variants B and C), split by power
# ============================================================
SMK = {'data': ('sig_mu_data', 'sig_g_data'), 'scott': ('sig_mu_data_scott', 'sig_g_data_scott')}
ZK = {'single': ('dmu_sigma_single', 'dg_sigma_single'),
      'data': ('dmu_sigma_data', 'dg_sigma_data'),
      'scott': ('dmu_sigma_data_scott', 'dg_sigma_data_scott')}

def _series(power):
    ts = [t for t in TABLE if t['exp'].startswith(power)]
    return sorted(ts, key=lambda t: int(t['exp'].split('Trans')[-1]))

VAR2 = [('data', 'dataset  sigma/sqrt(N)', '#1d3557', -0.14),
        ('scott', 'dataset + Scott bandwidth', '#e76f51', +0.14)]
fig2 = make_subplots(rows=2, cols=2, horizontal_spacing=0.11, vertical_spacing=0.18,
                     subplot_titles=['mu — 1nW', 'mu — 3nW', 'gamma — 1nW', 'gamma — 3nW'])
for ci, power in enumerate(POWERS, start=1):
    ts = _series(power)
    xlab = [t['exp'].split('Trans')[-1] for t in ts]
    for v, lab, col, dx in VAR2:
        xs = [i + dx for i in range(len(ts))]
        fig2.add_trace(go.Scatter(
            x=xs, y=[t['mu'] / t['mu_true'] for t in ts], mode='markers',
            marker=dict(color=col, size=8),
            error_y=dict(type='data', array=[t[SMK[v][0]] / t['mu_true'] for t in ts], color=col, thickness=1.2),
            name=lab, legendgroup=v, showlegend=(ci == 1)), row=1, col=ci)
        fig2.add_trace(go.Scatter(
            x=xs, y=[t['gamma'] / t['gamma_true'] for t in ts], mode='markers',
            marker=dict(color=col, size=8),
            error_y=dict(type='data', array=[t[SMK[v][1]] / t['gamma_true'] for t in ts], color=col, thickness=1.2),
            name=lab, legendgroup=v, showlegend=False), row=2, col=ci)
    for rr in (1, 2):
        fig2.update_xaxes(title_text='transmission (%)', row=rr, col=ci,
                          tickmode='array', tickvals=list(range(len(ts))), ticktext=xlab)
        fig2.update_yaxes(title_text='ratio to truth (marker) ±1σ (whisker)', row=rr, col=ci)
        fig2.add_hline(y=1.0, line=dict(color='#2a9d8f', dash='dash'), row=rr, col=ci)
fig2.update_layout(height=720, margin=dict(t=80, b=45),
                   title_text='23a — point estimate ± CRB whisker  (variants B and C; A omitted here — see FIG 2b)')
fig2.show(config=CONFIG_PLOT)

# ============================================================
# 23a — FIG 2b: coverage |Delta|/sigma for ALL three variants (log scale), split by power
# ============================================================
VAR3 = [('single', 'A: per-scan (existing)', '#adb5bd', -0.18),
        ('data', 'B: dataset sigma/sqrt(N)', '#1d3557', 0.0),
        ('scott', 'C: dataset + Scott', '#e76f51', +0.18)]
fig2b = make_subplots(rows=2, cols=2, horizontal_spacing=0.11, vertical_spacing=0.18,
                      subplot_titles=['mu — 1nW   |Δμ|/σμ', 'mu — 3nW   |Δμ|/σμ',
                                      'gamma — 1nW   |Δγ|/σγ', 'gamma — 3nW   |Δγ|/σγ'])
for ci, power in enumerate(POWERS, start=1):
    ts = _series(power)
    xlab = [t['exp'].split('Trans')[-1] for t in ts]
    for v, lab, col, dx in VAR3:
        xs = [i + dx for i in range(len(ts))]
        fig2b.add_trace(go.Scatter(x=xs, y=[t[ZK[v][0]] for t in ts], mode='markers',
                                   marker=dict(color=col, size=8), name=lab,
                                   legendgroup=v, showlegend=(ci == 1)), row=1, col=ci)
        fig2b.add_trace(go.Scatter(x=xs, y=[t[ZK[v][1]] for t in ts], mode='markers',
                                   marker=dict(color=col, size=8), name=lab,
                                   legendgroup=v, showlegend=False), row=2, col=ci)
    for rr in (1, 2):
        fig2b.add_hrect(y0=0.1, y1=1.0, fillcolor='#2a9d8f', opacity=0.10, line_width=0, row=rr, col=ci)
        fig2b.add_hrect(y0=1.0, y1=2.0, fillcolor='#e9c46a', opacity=0.14, line_width=0, row=rr, col=ci)
        fig2b.add_hline(y=2.0, line=dict(color='#e76f51', dash='dash'), row=rr, col=ci)
        fig2b.update_xaxes(title_text='transmission (%)', row=rr, col=ci,
                           tickmode='array', tickvals=list(range(len(ts))), ticktext=xlab)
        fig2b.update_yaxes(title_text='|Δ| / σ  (log)', type='log', range=[0.05, 100], row=rr, col=ci)
fig2b.update_layout(height=720, margin=dict(t=80, b=45),
                    title_text='23a — coverage: |Δ|/σ (log).  green band = within 1σ, amber = 1–2σ, above the dashed line = outside 2σ')



In [12]:
# ============================================================
# 23a — FIG 3: normalised trajectories vs iteration (per power)
# ============================================================
fig3 = make_subplots(rows=2, cols=2, horizontal_spacing=0.10, vertical_spacing=0.16,
                     subplot_titles=['1nW — mu/mu_true', '3nW — mu/mu_true',
                                     '1nW — gamma/gamma_true', '3nW — gamma/gamma_true'])
for ci, power in enumerate(POWERS, start=1):
    for r in [r for r in RESULTS if r['power'] == power]:
        steps = [h['step'] for h in r['history']]
        fig3.add_trace(go.Scatter(x=steps, y=[h['mu'] / r['mu_true'] for h in r['history']],
                                  mode='lines', name=r['exp'], showlegend=(ci == 1)), row=1, col=ci)
        fig3.add_trace(go.Scatter(x=steps, y=[h['gamma'] / r['gamma_true'] for h in r['history']],
                                  mode='lines', name=r['exp'], showlegend=False), row=2, col=ci)
    fig3.add_hline(y=1.0, line=dict(color='#2a9d8f', dash='dash'), row=1, col=ci)
    fig3.add_hline(y=1.0, line=dict(color='#2a9d8f', dash='dash'), row=2, col=ci)
fig3.update_xaxes(title_text='iteration'); fig3.update_yaxes(title_text='ratio')
fig3.update_layout(height=620, margin=dict(t=70, b=40),
                   title_text='23a — normalised trajectories (goal: 1.0)')
fig3.show(config=CONFIG_PLOT)


In [13]:
# ============================================================
# 23a — FIG 4: accuracy vs transmission (mu and gamma ratios, both powers, goal bands)
# ============================================================
fig4 = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
                     subplot_titles=['mu_hat/mu_true  (goal band 0.8-1.25)', 'gamma_hat/gamma_true  (goal band 0.8-1.25)'])
COL = {'1nW': '#1d3557', '3nW': '#e76f51'}
_seen_mu, _seen_g = set(), set()
for t in sorted(TABLE, key=lambda t: (t['exp'].split()[0], int(t['exp'].split('Trans')[-1]))):
    power = t['exp'].split()[0]
    x = int(t['exp'].split('Trans')[-1])
    sm, sg = power not in _seen_mu, power not in _seen_g
    _seen_mu.add(power); _seen_g.add(power)
    fig4.add_trace(go.Scatter(x=[x], y=[t['mu'] / t['mu_true']], mode='markers',
                              marker=dict(color=COL[power], size=9), name=power,
                              legendgroup=power, showlegend=sm, legendgrouptitle=dict(text='')), row=1, col=1)
    fig4.add_trace(go.Scatter(x=[x], y=[t['gamma'] / t['gamma_true']], mode='markers',
                              marker=dict(color=COL[power], size=9), name=power,
                              legendgroup=power, showlegend=False), row=1, col=2)
for cc in (1, 2):
    fig4.add_hrect(y0=0.8, y1=1.25, fillcolor='#2a9d8f', opacity=0.10, line_width=0, row=1, col=cc)
    fig4.add_hline(y=1.0, line=dict(color='#2a9d8f', dash='dash'), row=1, col=cc)
    fig4.update_xaxes(title_text='transmission (%)', row=1, col=cc)
fig4.update_layout(height=430, margin=dict(t=70, b=40), title_text='23a — accuracy vs transmission')
fig4.show(config=CONFIG_PLOT)


## Notes / caveats
- **Frozen model**: exp6 `trial_07` (`sigma_weight 0.8649`, `h_f_scale 4.0`, `h_s_scale 3.6745`,
  `gamma_rel_cap 0.10736`) on the exp5 schedule. Nothing changed in `src/` or previous notebooks.
- **Likelihood**: the 2-D `(FWHM, sigma_fit)` KDE; bandwidths **fixed from the target** (rule = Scott x
  scale) -> a clean likelihood in (mu, gamma). The μ-score is the exact REINFORCE score
  (`n ~ N(mu, sigma_prop^2)`); the Fisher uses the **raw** γ chain (`GAMMA_SCALE=False`).
- **Three σ variants**: (A) the **existing per-scan** convention `J = sum_i s_i s_i^T / N`;
  (B) the **dataset CRB** `sigma/sqrt(N)`; (C) the dataset CRB recomputed at the **Scott baseline**
  bandwidth (h=1) — to separate the uncertainty from the bandwidth exp6/7 inflated.
- **Known caveats carried in**: the target is filtered (`fit_err/FWHM < 10`) -> selection, so `N` is not
  a clean i.i.d. sample size; all `s_i` share one MC cloud (M_FINAL, averaged over FISHER_SEEDS);
  J is an **observed** (outer-product) Fisher; FM#8 means the μ-curvature comes from a mis-scaled
  σ_fit channel.
- **FIG 2 is split in two**: **2a** = point estimate ± the CRB whisker (dataset CRB: B and C; A omitted —
  its whiskers are ±4–5× the ratio and would flatten the panel), **2b** = coverage `|Δ|/σ` on a log axis for
  **all three** variants with the 1σ (green) / 2σ (amber) bands.
- The notebook is **cache-aware**: it loads `data/processed/23a_fisher_exp6_trial07.json` by default
  (re-render in seconds); `NB_RECOMPUTE=1` forces the full ~35 min run.
- Companion run data: `data/processed/23a_fisher_exp6_trial07.json`.